<a href="https://colab.research.google.com/github/guthasushmasree/ML-LabPrograms/blob/main/ml(skill_3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif

uploaded = files.upload()

csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]

if len(csv_files) == 0:
    print("Please upload a CSV file and run this cell again.")
else:
    file_name = csv_files[0]


    df = pd.read_csv(file_name)

    print("Dataset loaded successfully!")
    print("File:", file_name)
    print("Dataset Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())



    def find_column(possible_names):
        for name in possible_names:
            if name in df.columns:
                return name
        return None

    cgpa_col = find_column([
        "CGPA", "cgpa"
    ])

    attendance_col = find_column([
        "AttendancePercent", "attendance", "Attendance"
    ])

    backlog_col = find_column([
        "HistoryOfBacklogs", "backlogs", "Backlogs"
    ])

    internship_col = find_column([
        "Internships", "internships", "Internship"
    ])

    coding_col = find_column([
        "CodingTestScore", "coding_score", "CodingScore"
    ])

    communication_col = find_column([
        "SoftSkillsRating", "comm_score", "CommunicationScore",
        "CommunicationSkills"
    ])

    placement_col = find_column([
        "PlacementStatus", "placed", "Placed"
    ])

    print("\nDetected columns:")
    print("CGPA              :", cgpa_col)
    print("Attendance        :", attendance_col)
    print("Backlogs          :", backlog_col)
    print("Internships       :", internship_col)
    print("Coding Score      :", coding_col)
    print("Communication     :", communication_col)
    print("Placement         :", placement_col)


    if placement_col is not None:

        if df[placement_col].dtype == "object":

            df["placed"] = (
                df[placement_col]
                .astype(str)
                .str.strip()
                .str.lower()
                .map({
                    "placed": 1,
                    "yes": 1,
                    "true": 1,
                    "1": 1,
                    "not placed": 0,
                    "no": 0,
                    "false": 0,
                    "0": 0
                })
            )

        else:
            df["placed"] = pd.to_numeric(
                df[placement_col],
                errors="coerce"
            )

        df["placed"] = df["placed"].fillna(0).astype(int)



    if coding_col is not None and communication_col is not None:
        df["score_avg"] = (
            pd.to_numeric(df[coding_col], errors="coerce") +
            pd.to_numeric(df[communication_col], errors="coerce")
        ) / 2

    if backlog_col is not None:
        df["has_backlog"] = (
            pd.to_numeric(df[backlog_col], errors="coerce")
            .fillna(0) > 0
        ).astype(int)

    if cgpa_col is not None:
        df["cgpa_band"] = pd.cut(
            pd.to_numeric(df[cgpa_col], errors="coerce"),
            bins=[0, 6.5, 8, 10],
            labels=["Low", "Mid", "High"],
            include_lowest=True
        )

    if attendance_col is not None and cgpa_col is not None:
        df["att_x_cgpa"] = (
            pd.to_numeric(df[attendance_col], errors="coerce") *
            pd.to_numeric(df[cgpa_col], errors="coerce")
        )

    if internship_col is not None and "has_backlog" in df.columns:
        df["exp_index"] = (
            pd.to_numeric(df[internship_col], errors="coerce").fillna(0) * 2
            - df["has_backlog"]
        )



    engineered = [
        col for col in [
            "score_avg",
            "has_backlog",
            "cgpa_band",
            "att_x_cgpa",
            "exp_index"
        ]
        if col in df.columns
    ]

    print("\nEngineered Features:")
    print(df[engineered].head())

    print("\nFeature engineering completed successfully!")


    df.to_csv(
        "students_features.csv",
        index=False
    )

    print("\nSaved successfully as: students_features.csv")

Saving placement_predict_50k Dataset.csv to placement_predict_50k Dataset.csv
Dataset loaded successfully!
File: placement_predict_50k Dataset.csv
Dataset Shape: (50000, 32)

Columns:
['StudentID', 'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'CGPA_Tier', 'PlacementStatus', 'IsAnomaly', 'Salary Package']

Detected columns:
CGPA              : CGPA
Attendance        : AttendancePercent
Backlogs          : HistoryOfBacklogs
Internships       : Internships
Coding Score      : CodingTestScore
Communication     : SoftSkillsRating
Placement         : PlacementStatus

Engineered Features:
   score_avg  has_backlog cgpa_band  att_x_cgpa  exp_ind